In [4]:
from pathlib import Path
import numpy as np
import py3Dmol
from IPython.display import display


# Locate project and read XYZ files
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "xyz_files").is_dir()
)
XYZ = ROOT / "xyz_files"


def read_xyz(name):
    text = (XYZ / name).read_text()
    lines = text.splitlines()
    atoms = [
        (p[0], *map(float, p[1:4]))
        for p in map(str.split, lines[2:2 + int(lines[0])])
    ]
    return text, atoms


min_xyz, min_atoms = read_xyz("BH28_BHPERI_4_min.xyz")
ts_xyz, ts_atoms = read_xyz("BH28_BHPERI_4_ts.xyz")


# XYZ row -> consistent atom numbering
MIN = {
    1: "H1", 2: "C1", 3: "C2", 4: "C3", 5: "C4", 6: "C5",
    7: "H2", 8: "H6", 9: "H3", 10: "H4", 11: "H5"
}

TS = {
    1: "C1", 2: "C5", 3: "C2", 4: "C4", 5: "C3", 6: "H1",
    7: "H2", 8: "H6", 9: "H3", 10: "H5", 11: "H4"
}


def pos(atoms, mapping, label):
    i = next(i for i, lab in mapping.items() if lab == label) - 1
    return np.array(atoms[i][1:4], float)


def point(p):
    return dict(zip(("x", "y", "z"), map(float, p)))


def dist(atoms, mapping, a, b):
    return np.linalg.norm(
        pos(atoms, mapping, a) - pos(atoms, mapping, b)
    )


def add_bond(
    viewer, atoms, mapping, a, b, panel,
    color, radius=0.05, opacity=0.9
):
    viewer.addCylinder(
        {
            "start": point(pos(atoms, mapping, a)),
            "end": point(pos(atoms, mapping, b)),
            "radius": radius,
            "color": color,
            "opacity": opacity,
            "fromCap": True,
            "toCap": True,
        },
        viewer=panel,
    )


def add_distance(
    viewer, atoms, mapping, a, b, panel,
    color, offset, separator="···"
):
    p = (
        pos(atoms, mapping, a)
        + pos(atoms, mapping, b)
    ) / 2 + np.array(offset)

    viewer.addLabel(
        f"{a}{separator}{b} = {dist(atoms, mapping, a, b):.3f} Å",
        {
            "position": point(p),
            "fontColor": color,
            "backgroundColor": "white",
            "backgroundOpacity": 0.92,
            "fontSize": 13,
            "showBackground": True,
            "inFront": True,
        },
        viewer=panel,
    )


def add_atom_labels(viewer, atoms, mapping, panel):
    centre = np.mean([a[1:4] for a in atoms], axis=0)

    for label in [
        "C1", "C2", "C3", "C4", "C5",
        "H1", "H2", "H3", "H4", "H5", "H6"
    ]:
        p = pos(atoms, mapping, label)
        direction = p - centre
        direction /= np.linalg.norm(direction) or 1

        p += direction * (
            0.38 if label.startswith("H") else 0.28
        )

        viewer.addLabel(
            label,
            {
                "position": point(p),
                "fontColor":
                    "blue" if label.startswith("H") else "black",
                "backgroundColor": "white",
                "backgroundOpacity": 0.78,
                "fontSize": 11,
                "showBackground": True,
                "inFront": True,
            },
            viewer=panel,
        )


# Side-by-side viewer
viewer = py3Dmol.view(
    width=1250,
    height=600,
    viewergrid=(1, 2),
    linked=False,
)

style = {
    "stick": {"radius": 0.13},
    "sphere": {"scale": 0.25},
}


# ============================================================
# Reactant / Minimum
# ============================================================

left = (0, 0)

viewer.addModel(min_xyz, "xyz", viewer=left)
viewer.setStyle({}, style, viewer=left)
viewer.setBackgroundColor("white", viewer=left)

# C1-H1 bond
add_bond(
    viewer, min_atoms, MIN,
    "C1", "H1", left,
    "blue", 0.055, 0.95
)

# H1···C5 interaction
add_bond(
    viewer, min_atoms, MIN,
    "H1", "C5", left,
    "grey", 0.025, 0.60
)

add_distance(
    viewer, min_atoms, MIN,
    "C1", "H1", left,
    "blue",
    (0, 0, 0.35),
    separator="-"
)

add_distance(
    viewer, min_atoms, MIN,
    "H1", "C5", left,
    "grey",
    (0, 0, -0.35)
)

add_atom_labels(
    viewer, min_atoms, MIN, left
)

viewer.zoomTo(viewer=left)


# ============================================================
# Transition State
# ============================================================

right = (0, 1)

viewer.addModel(ts_xyz, "xyz", viewer=right)
viewer.setStyle({}, style, viewer=right)
viewer.setBackgroundColor("white", viewer=right)

# Breaking C1···H1 interaction
add_bond(
    viewer, ts_atoms, TS,
    "C1", "H1", right,
    "red"
)

# Forming C5···H1 interaction
add_bond(
    viewer, ts_atoms, TS,
    "C5", "H1", right,
    "red"
)

# Move C1···H1 label outward toward negative z
add_distance(
    viewer, ts_atoms, TS,
    "C1", "H1", right,
    "red",
    (0, 0, -0.48)
)

# Move C5···H1 label outward toward positive z
add_distance(
    viewer, ts_atoms, TS,
    "C5", "H1", right,
    "red",
    (0, 0, 0.48)
)

add_atom_labels(
    viewer, ts_atoms, TS, right
)

viewer.zoomTo(viewer=right)


display(viewer)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.